# ORCA-X v2.6 — Kaggle GPU Training

This is the controlled entry point for the ORCA-X forward-6-hour production-model candidate. It uses the canonical 2020–2025 historical dataset, runs the training preflight, evaluates the candidate on the locked 2025 temporal test and Digha spatial holdout, and **does not promote a model by default**.

**Kaggle settings:** Accelerator = GPU (Tesla T4 is sufficient), Internet = ON.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = 'main'
REPO_DIR = Path('/kaggle/working/HackHeritage')

# Safe defaults: CUDA enabled, two XGBoost workers, promotion disabled.
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'
os.environ['ORCA_PROMOTE_MODEL'] = 'false'

print('Repository:', REPO_URL)
print('Branch:', REPO_REF)
print('ORCA_X_DEVICE:', os.environ['ORCA_X_DEVICE'])
print('ORCA_X_N_JOBS:', os.environ['ORCA_X_N_JOBS'])
print('ORCA_PROMOTE_MODEL:', os.environ['ORCA_PROMOTE_MODEL'])

In [ ]:
%cd /kaggle/working
!rm -rf HackHeritage
!git clone --depth 1 --branch main --single-branch "{REPO_URL}" HackHeritage
%cd /kaggle/working/HackHeritage
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt

import torch
import xgboost as xgb

print('PyTorch CUDA available:', torch.cuda.is_available())
print('XGBoost version:', xgb.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
DATA = Path('/kaggle/working/HackHeritage/ml/data/processed/orca_historical_marine_risk.parquet')
if not DATA.exists():
    print('Canonical dataset missing; rebuilding from real Open-Meteo historical sources...')
    !python ml/src/colab_prepare.py
else:
    print('Canonical dataset already exists:', DATA)

assert DATA.exists(), DATA
print('Dataset ready:', DATA)

In [ ]:
# Mandatory fail-fast audit. This does not train or modify the production artifact.
!python ml/src/training_preflight.py

In [ ]:
# Run the v2.6 candidate evaluation. Promotion remains disabled.
!python ml/src/colab_gpu_runner.py ml/src/train.py

## Promotion gate

Do **not** set `ORCA_PROMOTE_MODEL=true` just because training completed. First inspect the generated evaluation report and verify the 2025 temporal test, Digha spatial holdout, critical-class recall/miss rate, and artifact/metadata agreement. The committed legacy production model must remain untouched until those gates pass.